# Phase 1 — Data Ingestion, Cleaning, and Validation

This notebook ingests the raw EIA, ERCOT, and Henry Hub data; standardizes it into clean tables; loads those tables into DuckDB; and runs the mandatory Phase 1 gate check.

In [1]:

import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from scipy import stats
import openpyxl
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Environment versions:")
print(f"  duckdb: {duckdb.__version__}")
print(f"  pandas: {pd.__version__}")
print(f"  numpy: {np.__version__}")
print(f"  statsmodels: {sm.__version__}")
print(f"  openpyxl: {openpyxl.__version__}")

import sys
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
conn = duckdb.connect('ercot_intelligence.db')
print("\nDuckDB connection: OK")

required_files = [
    PROJECT_ROOT / 'data/raw/eia_texas_generation.csv',
    PROJECT_ROOT / 'data/raw/Henry_Hub_Natural_Gas_Spot_Price.csv',
]
print("\nRequired raw files:")
for file_path in required_files:
    exists = file_path.exists()
    size_kb = file_path.stat().st_size / 1024 if exists else 0
    print(f"  {file_path.relative_to(PROJECT_ROOT)}: {'OK' if exists else 'MISSING'} ({size_kb:.1f} KB)")
conn.close()


Environment versions:
  duckdb: 1.5.3
  pandas: 3.0.3
  numpy: 2.4.6
  statsmodels: 0.14.6
  openpyxl: 3.1.5

DuckDB connection: OK

Required raw files:
  data/raw/eia_texas_generation.csv: OK (48.0 KB)
  data/raw/Henry_Hub_Natural_Gas_Spot_Price.csv: OK (119.7 KB)


In [2]:

from scripts import phase1_ingestion as phase1

gen_clean = phase1.clean_eia_generation()
gen_clean.to_csv(PROJECT_ROOT / 'data/processed/generation_clean.csv', index=False)
print(f"Saved generation_clean.csv: {len(gen_clean):,} rows")
print("\nFirst 10 cleaned generation rows:")
print(gen_clean.head(10).to_string(index=False))


EIA Texas rows (fuel series): 20
['Texas : all fuels (utility-scale)', 'Texas : coal', 'Texas : petroleum liquids', 'Texas : petroleum coke', 'Texas : natural gas', 'Texas : other gases', 'Texas : nuclear', 'Texas : conventional hydroelectric', 'Texas : other renewables', 'Texas : wind', 'Texas : all utility-scale solar', 'Texas : geothermal', 'Texas : biomass', 'Texas : wood and wood-derived fuels', 'Texas : other biomass', 'Texas : hydro-electric pumped storage', 'Texas : other', 'Texas : all solar', 'Texas : small-scale solar photovoltaic', 'Texas : all utility-scale solar']
Generation clean shape: (840, 6)
Year range: 2015–2024
Fuel types: ['coal', 'hydro', 'natural_gas', 'nuclear', 'other', 'solar', 'wind']
Saved generation_clean.csv: 840 rows

First 10 cleaned generation rows:
      date  year  month   fuel_type  generation_mwh  pct_of_total_monthly
2015-01-01  2015      1        coal      11365000.0               29.7794
2015-01-01  2015      1 natural_gas      19252000.0       

In [3]:

load_clean = phase1.clean_ercot_load()
load_clean.to_csv(PROJECT_ROOT / 'data/processed/load_clean.csv', index=False)
print(f"Saved load_clean.csv: {len(load_clean):,} rows")
print("\nFirst 10 cleaned load rows:")
print(load_clean.head(10).to_string(index=False))


  Skipping 2007_ercot_hourly_load_data.xls: outside core analysis window 2015-2024
  Skipping 2008_ercot_hourly_load_data.xls: outside core analysis window 2015-2024
  Skipping 2009_ercot_hourly_load_data.xls: outside core analysis window 2015-2024
  Skipping 2010_ercot_hourly_load_data.xls: outside core analysis window 2015-2024
  Skipping 2011_ercot_hourly_load_data.xls: outside core analysis window 2015-2024
  Skipping 2012_ercot_hourly_load_data.xls: outside core analysis window 2015-2024
  Skipping 2013_ercot_hourly_load_data.xls: outside core analysis window 2015-2024
  Skipping 2014_ercot_hourly_load_data.xls: outside core analysis window 2015-2024


  Year 2018: 78,840 rows, zones: ['COAST', 'EAST', 'FWEST', 'NCENT', 'NORTH', 'SCENT', 'SOUTH', 'TOTAL', 'WEST']


  Year 2019: 78,840 rows, zones: ['COAST', 'EAST', 'FWEST', 'NCENT', 'NORTH', 'SCENT', 'SOUTH', 'TOTAL', 'WEST']


  Year 2020: 79,056 rows, zones: ['COAST', 'EAST', 'FWEST', 'NCENT', 'NORTH', 'SCENT', 'SOUTH', 'TOTAL', 'WEST']


  Year 2021: 78,840 rows, zones: ['COAST', 'EAST', 'FWEST', 'NCENT', 'NORTH', 'SCENT', 'SOUTH', 'TOTAL', 'WEST']


...

  Year 2023: 78,840 rows, zones: ['COAST', 'EAST', 'FWEST', 'NCENT', 'NORTH', 'SCENT', 'SOUTH', 'TOTAL', 'WEST']


  Year 2024: 79,056 rows, zones: ['COAST', 'EAST', 'FWEST', 'NCENT', 'NORTH', 'SCENT', 'SOUTH', 'TOTAL', 'WEST']
  Skipping Native_Load_2025.xlsx: outside core analysis window 2015-2024


  Year 2016: 79,047 rows, zones: ['COAST', 'EAST', 'FWEST', 'NCENT', 'NORTH', 'SCENT', 'SOUTH', 'TOTAL', 'WEST']


  Year 2017: 78,840 rows, zones: ['COAST', 'EAST', 'FWEST', 'NCENT', 'NORTH', 'SCENT', 'SOUTH', 'TOTAL', 'WEST']
  Year 2015: 78,840 rows, zones: ['COAST', 'EAST', 'FWEST', 'NCENT', 'NORTH', 'SCENT', 'SOUTH', 'TOTAL', 'WEST']

Combined load: (789039, 7)
Date range: 2015-01-01 01:00:00 to 2024-12-31 23:00:00
Unique zones: ['COAST', 'EAST', 'FWEST', 'NCENT', 'NORTH', 'SCENT', 'SOUTH', 'TOTAL', 'WEST']
Load MW min/max: 488 / 85,464
Negative values: 0, >90,000 MW: 0


Saved load_clean.csv: 789,039 rows

First 10 cleaned load rows:
           datetime  year  month  day  hour  zone      load_mw
2015-01-01 01:00:00  2015      1    1     1  WEST  1470.814721
2015-01-01 01:00:00  2015      1    1     1 SCENT  6731.301663
2015-01-01 01:00:00  2015      1    1     1 NORTH   901.770195
2015-01-01 01:00:00  2015      1    1     1 SOUTH  3607.904503
2015-01-01 01:00:00  2015      1    1     1  EAST  1350.784678
2015-01-01 01:00:00  2015      1    1     1 COAST  9844.200268
2015-01-01 01:00:00  2015      1    1     1 TOTAL 39624.861027
2015-01-01 01:00:00  2015      1    1     1 NCENT 13640.024978
2015-01-01 01:00:00  2015      1    1     1 FWEST  2078.060021
2015-01-01 02:00:00  2015      1    1     2 NCENT 13425.121941


In [4]:

gas_clean = phase1.clean_gas_prices()
gas_clean.to_csv(PROJECT_ROOT / 'data/processed/gas_prices_clean.csv', index=False)
print(f"Saved gas_prices_clean.csv: {len(gas_clean):,} rows")
print("\nFirst 10 cleaned gas price rows:")
print(gas_clean.head(10).to_string(index=False))


Gas prices monthly rows: 120
Date range: 2015-01-30 to 2024-12-31
Price range: $1.49 – $8.81/MMBtu
Saved gas_prices_clean.csv: 120 rows

First 10 cleaned gas price rows:
 year  month       date  monthly_avg  monthly_min  monthly_max
 2015      1 2015-01-30     2.994500         2.88         3.32
 2015      2 2015-02-27     2.874737         2.62         3.22
 2015      3 2015-03-31     2.830909         2.64         3.27
 2015      4 2015-04-30     2.609048         2.50         2.72
 2015      5 2015-05-29     2.850000         2.68         3.04
 2015      6 2015-06-30     2.783636         2.60         2.94
 2015      7 2015-07-31     2.839565         2.70         2.93
 2015      8 2015-08-31     2.773810         2.65         2.93
 2015      9 2015-09-30     2.660909         2.47         2.74
 2015     10 2015-10-30     2.340909         1.98         2.54


In [5]:

uri_load, baseline_feb = phase1.build_uri_tables(load_clean)
uri_load.to_csv(PROJECT_ROOT / 'data/processed/storm_uri_clean.csv', index=False)
print(f"Saved storm_uri_clean.csv: {len(uri_load):,} rows")
print("\nFirst 10 Uri rows:")
print(uri_load.head(10).to_string(index=False))
print("\nBaseline February sample:")
print(baseline_feb.head(10).to_string(index=False))


Uri period rows: 6048
Baseline Feb rows: 12312
Uri peak load (TOTAL): 69,692 MW
Uri min load (TOTAL): 27,800 MW
Baseline Feb avg load (TOTAL): 39,540 MW
Saved storm_uri_clean.csv: 6,048 rows

First 10 Uri rows:
           datetime  year  month  day  hour  zone      load_mw
2021-02-01 00:00:00  2021      2    1     0 NORTH   754.305556
2021-02-01 00:00:00  2021      2    1     0 NCENT 12083.923290
2021-02-01 00:00:00  2021      2    1     0 FWEST  3743.417130
2021-02-01 00:00:00  2021      2    1     0  EAST  1564.850771
2021-02-01 00:00:00  2021      2    1     0 COAST 10866.331202
2021-02-01 00:00:00  2021      2    1     0 SOUTH  2741.154265
2021-02-01 00:00:00  2021      2    1     0 TOTAL 38807.021769
2021-02-01 00:00:00  2021      2    1     0  WEST  1094.401408
2021-02-01 00:00:00  2021      2    1     0 SCENT  5958.638147
2021-02-01 01:00:00  2021      2    1     1 SCENT  5263.169851

Baseline February sample:
           datetime  year  month  day  hour  zone      load_mw
2019-0

In [6]:

conn = phase1.load_duckdb(gen_clean, load_clean, gas_clean, uri_load)
print("\nDuckDB table preview — ercot_generation:")
print(conn.execute('SELECT * FROM ercot_generation ORDER BY year, month, fuel_type LIMIT 10').df().to_string(index=False))
print("\nDuckDB table preview — ercot_load TOTAL:")
print(conn.execute("SELECT * FROM ercot_load WHERE zone = 'TOTAL' ORDER BY datetime LIMIT 10").df().to_string(index=False))



DuckDB tables created:
  ercot_generation: 840 rows
  ercot_load: 789,039 rows
  gas_prices: 120 rows
  storm_uri_load: 6,048 rows

DuckDB table preview — ercot_generation:


      date  year  month   fuel_type  generation_mwh  pct_of_total_monthly
2015-01-01  2015      1        coal      11365000.0               29.7794
2015-01-01  2015      1       hydro        131000.0                0.3433
2015-01-01  2015      1 natural_gas      19252000.0               50.4454
2015-01-01  2015      1     nuclear       3817000.0               10.0016
2015-01-01  2015      1       other        536000.0                1.4045
2015-01-01  2015      1       solar         32000.0                0.0838
2015-01-01  2015      1        wind       3031000.0                7.9420
2015-02-01  2015      2        coal       9146000.0               27.3431
2015-02-01  2015      2       hydro          8000.0                0.0239
2015-02-01  2015      2 natural_gas      16980000.0               50.7638

DuckDB table preview — ercot_load TOTAL:
           datetime  year  month  day  hour  zone      load_mw
2015-01-01 01:00:00  2015      1    1     1 TOTAL 39624.861027
2015-01-01 02:00:0

In [7]:

validation_results = phase1.run_validation(conn)

report_path = PROJECT_ROOT / 'outputs/data_quality_report.txt'
with open(report_path, 'w') as report_file:
    report_file.write('ERCOT Power Market Intelligence — Data Quality Report\n')
    report_file.write('=' * 60 + '\n\n')
    for title, result in validation_results:
        report_file.write(title + '\n')
        report_file.write('-' * len(title) + '\n')
        if isinstance(result, pd.DataFrame):
            report_file.write(result.to_string(index=False) + '\n\n')
        else:
            report_file.write(str(result) + '\n\n')
    report_file.write('Report generated successfully\n')

print(f"Data quality report saved to {report_path}")


      table_name   earliest     latest  total_rows  years_covered
ercot_generation 2015-01-01 2024-12-01         840             10
table_name            earliest              latest  total_rows  years_covered
ercot_load 2015-01-01 01:00:00 2024-12-31 23:00:00      789039             10
table_name   earliest     latest  total_rows  years_covered
gas_prices 2015-01-30 2024-12-31         120             10

Null check — ercot_generation:
 null_generation  null_fuel_type  null_year  null_pct
             0.0             0.0        0.0       0.0

Fuel type distribution:
  fuel_type  months  total_twh  avg_monthly_share_pct
natural_gas     120    2474.80                  49.55
       coal     120     971.12                  19.95
       wind     120     879.58                  18.00
    nuclear     120     405.05                   8.35
      solar     120     144.33                   2.69
      other     120      59.83                   1.24
      hydro     120      10.27                   

In [8]:

phase1_passed = phase1.run_gate_check(conn)
conn.close()

print("""
Phase 1 gate check complete.
Please review all outputs above including:
- Data quality report at outputs/data_quality_report.txt
- Row counts and date ranges for all 4 tables
- Fuel type distribution table
- All validation check results

Type APPROVED to proceed to Phase 2, or describe any issues to fix.
""")


PHASE 1 GATE CHECK

Gate check results:
  [PASS] all_tables_exist
  [PASS] generation_year_range
  [PASS] all_fuel_types
  [PASS] no_null_generation
  [PASS] load_year_coverage
  [PASS] gas_prices_loaded
  [PASS] uri_data_loaded
  [PASS] processed_files_saved
  [PASS] quality_report_saved

ALL PHASE 1 CHECKS PASSED
  ercot_generation: 840 rows
  ercot_load: 789,039 rows
  gas_prices: 120 rows
  storm_uri_load: 6,048 rows
  Fuel types: ['coal', 'hydro', 'natural_gas', 'nuclear', 'other', 'solar', 'wind']
  Generation year range: 2015–2024
  Load year range: 2015–2024

Phase 1 gate check complete.
Please review all outputs above including:
- Data quality report at outputs/data_quality_report.txt
- Row counts and date ranges for all 4 tables
- Fuel type distribution table
- All validation check results

Type APPROVED to proceed to Phase 2, or describe any issues to fix.

